## Portfolio Performance

Use this notebook to calculate your performance compared to major indices

In [1]:
import numpy as np
import pandas as pd

In [2]:
import os

# Get the current working directory
current_directory = os.getcwd()
print(f"Current Working Directory: {current_directory}")

# Change the working directory
new_directory = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature/code/Users/omai.r/spectral_nature/src/rh_perf/src'
os.chdir(new_directory)
print(f"Changed Working Directory to: {new_directory}")

Current Working Directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/src


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature/code/Users/omai.r/spectral_nature/src/rh_perf/src'

In [1]:
import getpass

import robin_stocks.robinhood as r

def robinhood_login(username, password):
    """
    Logs into Robinhood account using provided username and password.
    
    Parameters:
    username (str): Robinhood account username
    password (str): Robinhood account password
    
    Returns:
    dict: Login result containing access token and other details
    """
    login_result = r.login(username, password)
    return login_result

# Example usage:
# login_result = robinhood_login(rh_username, rh_password)
# print(login_result)
# Prompt for username and password

rh_username = input("Enter your Robinhood username: ")
rh_password = getpass.getpass("Enter your Robinhood password: ")

# Login to Robinhood
login_result = robinhood_login(rh_username, rh_password)
#print(login_result)

Starting login process...


In [2]:
import robin_stocks.robinhood as r
from robin_stocks.robinhood import authentication
from urllib.parse import urlencode

def get_account_number():
    prof = r.profiles.load_account_profile()
    return prof.get("account_number")

def try_portfolio_historicals(account_number=None,
                              intervals=("week","day"),
                              spans=("5year","year","3month","all","month"),
                              bounds=("regular","extended")):
    if account_number is None:
        account_number = get_account_number()

    session = authentication.get_session()
    base_url = "https://api.robinhood.com/portfolios/historicals/{}/".format(account_number)

    results = []
    for interval in intervals:
        for span in spans:
            for b in bounds:
                params = dict(interval=interval, span=span, bounds=b)
                url = base_url + "?" + urlencode(params)
                resp = session.get(url, timeout=15)
                print(f"{resp.status_code} {url}")
                if resp.status_code == 200:
                    j = resp.json()
                    results.append((url, j))
                else:
                    # show brief error body
                    try:
                        print(resp.text[:200])
                    except Exception:
                        pass
    return results

def fetch_portfolio_historicals(account_number=None,
                                interval="week",
                                span="5year",
                                bounds="regular"):
    if account_number is None:
        account_number = get_account_number()
    session = authentication.get_session()
    url = f"https://api.robinhood.com/portfolios/historicals/{account_number}/"
    params = dict(interval=interval, span=span, bounds=bounds)
    resp = session.get(url, params=params, timeout=15)
    if resp.status_code == 404:
        raise ValueError(f"404 for {resp.url}. Verify account_number and allowed interval/span combo.")
    resp.raise_for_status()
    return resp.json()

In [4]:
import robin_stocks.robinhood as r

rh_username = input("Enter your Robinhood username: ")
rh_password = getpass.getpass("Enter your Robinhood password: ")
r.login(rh_username, rh_password)  # plus MFA if needed


# Scan for any working combination (prints status codes)
results = try_portfolio_historicals()

# If something worked, inspect one
if results:
    url, data = results[0]
    print("Using:", url)
    print("Points:", len(data.get("equity_historicals", [])))

# Direct single call (your original desired combo)
data = fetch_portfolio_historicals(interval="week", span="5year", bounds="regular")

Starting login process...


AttributeError: module 'robin_stocks.robinhood.authentication' has no attribute 'get_session'

In [15]:
import os
from azure.keyvault.secrets import SecretClient
from azure.identity import DefaultAzureCredential

KV_NAME = "spectral-nature-kvault"
KVUri = f"https://{KV_NAME}.vault.azure.net"
credential = DefaultAzureCredential()
client = SecretClient(vault_url=KVUri, credential=credential)

rh_username_value = client.get_secret("rh-username").value
rh_passwd_value = client.get_secret("rh-pswd").value



